<a href="https://colab.research.google.com/github/GaganKI/ExplainSomatic/blob/main/ExplainSomatic_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ExplainSomatic — Production Training (Colab GPU)

This notebook runs the real pipeline end to end: real SEQC2 (HCC1395) tumour-only
BAM data → candidate generation → cached tensors → production training of the
dual-stream CNN+Transformer model → Grad-CAM explainability check.

**Compute budget reality check:** full HCC1395 WGS at 1,500x depth is hundreds
of GB across all chromosomes. On a ~100-compute-unit Colab Pro budget, start
with **chromosome 21** (the smallest autosome) end-to-end, confirm the whole
pipeline and metrics look right, *then* scale to more chromosomes with the
same scripts (nothing below is chr21-specific except the `--region` arguments).

Runtime → Change runtime type → **GPU (T4 or better)** before running anything.


In [1]:
# 1. Mount Drive (checkpoints + caches persist here across sessions)
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/ExplainSomatic'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/ref', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/data', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/cache', exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/ckpts', exist_ok=True)
print('project dir:', PROJECT_DIR)


Mounted at /content/drive
project dir: /content/drive/MyDrive/ExplainSomatic


In [2]:
# 2. System + Python dependencies
!apt-get -qq install -y samtools sra-toolkit bwa tabix bcftools > /dev/null
!pip -q install pysam h5py

import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 17.0 MB/s eta 0:00:00
torch 2.11.0+cu128 | CUDA available: True
Tesla T4


In [3]:
# 3. Get the code
# Push models.py, losses.py, gradcam.py, candidate_generation.py, real_data.py,
# train_production.py to a GitHub repo first (e.g. GaganKI/ExplainSomatic), then:

!git clone https://github.com/GaganKI/ExplainSomatic.git /content/ExplainSomatic
%cd /content/ExplainSomatic

# --- OR, if you haven't pushed to GitHub yet, upload the 6 .py files directly: ---
#from google.colab import files
#uploaded = files.upload()  # select: models.py, losses.py, gradcam.py,
                            #         candidate_generation.py, real_data.py, train_production.py


Cloning into '/content/ExplainSomatic'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (17/17), done.
remote: Total 24 (delta 8), reused 22 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 18.37 KiB | 329.00 KiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/ExplainSomatic


In [16]:
!head -n 10 /content/drive/MyDrive/ExplainSomatic/cache/candidates_chr21.bed

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chrom	pos	ref_base	depth	alt_count	alt_frac
chr21	5030758	A	34	2	0.0588
chr21	5049385	C	13	5	0.3846
chr21	5063496	C	18	2	0.1111
chr21	5066422	C	21	2	0.0952
chr21	5066672	C	18	4	0.2222
chr21	5101941	T	43	2	0.0465
chr21	5103357	C	24	2	0.0833
chr21	5103364	C	24	2	0.0833
chr21	5103366	C	24	2	0.0833


## 4. Reference genome

This is the **exact same reference build the SEQC2 consortium and GDC use**
(`GRCh38.d1.vd1`), pulled directly from the GDC reference files API — a real,
working, verified download (875 MB), not a placeholder.


In [5]:
REF_DIR = f'{PROJECT_DIR}/ref'
REF_FA = f'{REF_DIR}/GRCh38.d1.vd1.fa'

if not os.path.exists(REF_FA):
    !curl -L -o {REF_DIR}/GRCh38.d1.vd1.fa.tar.gz \
        'https://api.gdc.cancer.gov/data/254f697d-310d-4d7d-a27b-27fbf767a834'
    !tar -xzf {REF_DIR}/GRCh38.d1.vd1.fa.tar.gz -C {REF_DIR}
    !samtools faidx {REF_FA}

print('reference ready:', os.path.exists(REF_FA + '.fai'))


reference ready: True


## 5. SEQC2 tumour-only BAM and truth VCF

Two real, citable sources (from the SEQC2 consortium's own site, not a guess):

- **Sequencing data (raw + some pre-aligned BAMs):**
  https://sites.google.com/view/seqc2/home/sequencing — raw FASTQs are on
  **SRA accession SRP162370**; some BWA-MEM aligned BAMs are on NCBI's FTP,
  linked from that page.
- **High-confidence truth VCF (somatic SNV + INDEL):**
  https://sites.google.com/view/seqc2/home/data-analysis/high-confidence-somatic-snv-and-indel-v1-2

Open both pages, copy the current direct link for the **HCC1395 tumour-only
BAM** (or the SRA run accession if only FASTQ is available) and the
**truth VCF**, and paste them into the cell below.

**On size:** a full SEQC2 WGS BAM is ~90-150 GB per replicate (63 tumor-normal
pairs exist in total -- you only need one). The cell below does **not**
download the full file. It uses `samtools view` on the remote URL directly,
which pulls only chr21's reads over HTTPS via range requests, so you never
touch the other ~40 chromosomes' worth of data. If the host doesn't support
this (rare for NCBI-hosted files, but possible for some mirrors), a fallback
full-download path is commented in the cell.


In [6]:
BAM_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/seqc/Somatic_Mutation_WG/data/WGS/WGS_IL_T_1.bwa.dedup.bam"
TRUTH_VCF_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/seqc/Somatic_Mutation_WG/release/latest/high-confidence_sSNV_in_HC_regions_v1.2.1.vcf.gz"
REGIONS_BED_URL = "https://ftp-trace.ncbi.nlm.nih.gov/ReferenceSamples/seqc/Somatic_Mutation_WG/release/latest/High-Confidence_Regions_v1.2.bed"

DATA_DIR = f'{PROJECT_DIR}/data'
BAM_PATH = f'{DATA_DIR}/HCC1395_tumor_chr21.bam'
TRUTH_VCF = f'{DATA_DIR}/seqc2_truth.vcf.gz'
REGIONS_BED = f'{DATA_DIR}/High-Confidence_Regions_v1.2.bed'

!samtools view -b "$BAM_URL" chr21 -o {BAM_PATH}
!samtools index {BAM_PATH}
!samtools idxstats {BAM_PATH} | awk '$3+$4 > 0'
!ls -lh {BAM_PATH}

!curl -L -o {TRUTH_VCF} "$TRUTH_VCF_URL"
!tabix -p vcf {TRUTH_VCF}

!curl -L -o {REGIONS_BED} "$REGIONS_BED_URL"

print('chr21-only BAM, truth VCF, and high-confidence regions BED all ready.')

chr21	46709983	22948739	92092
-rw------- 1 root root 1.4G Sep 12 13:34 /content/drive/MyDrive/ExplainSomatic/data/HCC1395_tumor_chr21.bam
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4705k  100 4705k    0     0  4128k      0  0:00:01  0:00:01 --:--:-- 4130k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 16.0M  100 16.0M    0     0  12.1M      0  0:00:01  0:00:01 --:--:-- 12.1M
chr21-only BAM, truth VCF, and high-confidence regions BED all ready.


## 6. Candidate generation (production prefilter, chr21)

This is the step that makes the whole thing tractable — instead of running the
deep model on all ~46M positions on chr21, we only keep positions with any
mismatch signal at all. `--min_alt_frac 0.01` is set deliberately near the 1%
floor of Objective 2's VAF target, so we don't accidentally filter out the
exact low-VAF examples the project cares about before the model even sees them.


In [7]:
CACHE_DIR = f'{PROJECT_DIR}/cache'
CANDIDATES_BED = f'{CACHE_DIR}/candidates_chr21.bed'

!python3 candidate_generation.py \
    --bam {BAM_PATH} --ref {REF_FA} --region chr21 \
    --out {CANDIDATES_BED} --min_alt_reads 2 --min_alt_frac 0.01 --min_depth 8 \
    --regions_bed {REGIONS_BED}

  (109958 candidates dropped for falling outside the high-confidence regions BED)
191306 candidate sites written to /content/drive/MyDrive/ExplainSomatic/cache/candidates_chr21.bed


In [4]:
PROJECT_DIR = '/content/drive/MyDrive/ExplainSomatic'
REF_FA = f'{PROJECT_DIR}/ref/GRCh38.d1.vd1.fa'
DATA_DIR = f'{PROJECT_DIR}/data'
BAM_PATH = f'{DATA_DIR}/HCC1395_tumor_chr21.bam'
TRUTH_VCF = f'{DATA_DIR}/seqc2_truth.vcf.gz'
REGIONS_BED = f'{DATA_DIR}/High-Confidence_Regions_v1.2.bed'
CACHE_DIR = f'{PROJECT_DIR}/cache'
CANDIDATES_BED = f'{CACHE_DIR}/candidates_chr21.bed'

!ls -la {REF_FA} {BAM_PATH} {TRUTH_VCF} {REGIONS_BED} {CANDIDATES_BED}

-rw------- 1 root root    5520310 Sep 12 14:13 /content/drive/MyDrive/ExplainSomatic/cache/candidates_chr21.bed
-rw------- 1 root root 1462471679 Sep 12 13:31 /content/drive/MyDrive/ExplainSomatic/data/HCC1395_tumor_chr21.bam
-rw------- 1 root root   16879606 Sep 12 13:34 /content/drive/MyDrive/ExplainSomatic/data/High-Confidence_Regions_v1.2.bed
-rw------- 1 root root    4818105 Sep 12 13:34 /content/drive/MyDrive/ExplainSomatic/data/seqc2_truth.vcf.gz
-rw------- 1 root root 3152288848 Apr 26  2016 /content/drive/MyDrive/ExplainSomatic/ref/GRCh38.d1.vd1.fa


## 7. Build cached tensors (train/val/test split by genomic region)

Splitting by **position range within chr21** (not a random shuffle) avoids
leaking near-identical pileup windows between train and val — two overlapping
windows 5bp apart from the same region are not independent examples.


In [5]:
import csv

rows = list(csv.DictReader(open(CANDIDATES_BED), delimiter='\t'))
rows.sort(key=lambda r: int(r['pos']))
n = len(rows)
train_rows, val_rows, test_rows = rows[:int(n*0.7)], rows[int(n*0.7):int(n*0.85)], rows[int(n*0.85):]

def write_bed(path, rows):
    with open(path, 'w') as f:
        f.write('chrom\tpos\tref_base\tdepth\talt_count\talt_frac\n')
        for r in rows:
            f.write('\t'.join(r[k] for k in ['chrom','pos','ref_base','depth','alt_count','alt_frac']) + '\n')

write_bed(f'{CACHE_DIR}/train.bed', train_rows)
write_bed(f'{CACHE_DIR}/val.bed', val_rows)
write_bed(f'{CACHE_DIR}/test.bed', test_rows)
print(f'train={len(train_rows)}  val={len(val_rows)}  test={len(test_rows)}')

NameError: name 'CANDIDATES_BED' is not defined

In [8]:
for split in ['train', 'val', 'test']:
    !python3 real_data.py \
        --candidates {CACHE_DIR}/{split}.bed \
        --bam {BAM_PATH} --ref {REF_FA} --truth_vcf {TRUTH_VCF} \
        --out {CACHE_DIR}/{split}.h5


starting new cache: /content/drive/MyDrive/ExplainSomatic/cache/train.h5, 133914 candidates
  cached 200/133914 candidates... (flushed)
  cached 400/133914 candidates... (flushed)
  cached 600/133914 candidates... (flushed)
  cached 800/133914 candidates... (flushed)
  cached 1000/133914 candidates... (flushed)
  cached 1200/133914 candidates... (flushed)
  cached 1400/133914 candidates... (flushed)
  cached 1600/133914 candidates... (flushed)
  cached 1800/133914 candidates... (flushed)
  cached 2000/133914 candidates... (flushed)
  cached 2200/133914 candidates... (flushed)
  cached 2400/133914 candidates... (flushed)
  cached 2600/133914 candidates... (flushed)
  cached 2800/133914 candidates... (flushed)
  cached 3000/133914 candidates... (flushed)
  cached 3200/133914 candidates... (flushed)
  cached 3400/133914 candidates... (flushed)
  cached 3600/133914 candidates... (flushed)
  cached 3800/133914 candidates... (flushed)
  cached 4000/133914 candidates... (flushed)
  cached 420

In [5]:
!mkdir -p /content/cache
!cp {CACHE_DIR}/train.h5 {CACHE_DIR}/val.h5 {CACHE_DIR}/test.h5 /content/cache/

In [6]:
!python3 repair_vaf.py --h5 /content/cache/train.h5
!python3 repair_vaf.py --h5 /content/cache/val.h5
!python3 repair_vaf.py --h5 /content/cache/test.h5

/content/cache/train.h5: repaired 124833/133914 vaf values
  positives now split: low-VAF(<5%)=10  high-VAF(>=5%)=294
/content/cache/val.h5: repaired 28672/28696 vaf values
  positives now split: low-VAF(<5%)=0  high-VAF(>=5%)=96
/content/cache/test.h5: repaired 28401/28696 vaf values
  positives now split: low-VAF(<5%)=1  high-VAF(>=5%)=64


In [9]:
!zcat {TRUTH_VCF} | grep -v '^#' | cut -f1 | sort | uniq -c | sort -rn

   3608 chr2
   3440 chr1
   3057 chr7
   3006 chr4
   2743 chr3
   2630 chr5
   2605 chr8
   1747 chr12
   1712 chr10
   1656 chr9
   1599 chr6
   1594 chr11
   1574 chr14
   1250 chr15
   1144 chr13
   1116 chr18
   1106 chr20
   1079 chr17
    874 chr19
    808 chr16
    627 chr22
    472 chr21


## 8. Train (full spec: 6-layer/8-head transformer, VAF-aware loss)

Checkpoints save every epoch to Drive, so a dropped Colab session doesn't
cost you compute units to redo — re-run this same cell with `--resume` and it
picks up where it left off.


In [7]:
CKPT_DIR = f'{PROJECT_DIR}/ckpts/fusion_vafaware_v3'

!python3 train_production.py \
    --train_h5 /content/cache/train.h5 --val_h5 /content/cache/val.h5 \
    --arch fusion --loss vaf_aware --transformer_layers 6 \
    --epochs 5 --batch_size 64 --lr 1e-3 --num_workers 0 \
    --ckpt_dir {CKPT_DIR}

device: cuda
train set: 304 positives, 133610 negatives (0.227% positive rate)
/content/ExplainSomatic/train_production.py:157: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
epoch 1/5  loss=0.4782  val_f1=0.007  low_vaf_recall=nan (n=0)  high_vaf_recall=1.000 (n=96)  [137.1s]
epoch 2/5  loss=0.3360  val_f1=0.029  low_vaf_recall=nan (n=0)  high_vaf_recall=0.906 (n=96)  [141.9s]
epoch 3/5  loss=0.2758  val_f1=0.031  low_vaf_recall=nan (n=0)  high_vaf_recall=0.875 (n=96)  [141.3s]
epoch 4/5  loss=0.2230  val_f1=0.014  low_vaf_recall=nan (n=0)  high_vaf_recall=0.885 (n=96)  [141.3s]
epoch 5/5  loss=0.1855  val_f1=0.027  low_vaf_recall=nan (n=0)  high_vaf_recall=0.875 (n=96)  [141.3s]
training complete. best low-VAF recall: -1.0


## 9. Ablation runs (to actually demonstrate the novelty claim)

Same data, same VAF-aware loss, different architectures — this is what turns
"we think fusion helps" into a number you can show a panel.


In [ ]:
!python3 train_production.py --train_h5 {CACHE_DIR}/train.h5 --val_h5 {CACHE_DIR}/val.h5 \
    --arch cnn_only --loss vaf_aware --epochs 30 --batch_size 64 \
    --ckpt_dir {PROJECT_DIR}/ckpts/cnn_only --resume

!python3 train_production.py --train_h5 {CACHE_DIR}/train.h5 --val_h5 {CACHE_DIR}/val.h5 \
    --arch transformer_only --loss vaf_aware --transformer_layers 6 --epochs 30 --batch_size 64 \
    --ckpt_dir {PROJECT_DIR}/ckpts/transformer_only --resume

!python3 train_production.py --train_h5 {CACHE_DIR}/train.h5 --val_h5 {CACHE_DIR}/val.h5 \
    --arch fusion --loss plain_bce --transformer_layers 6 --epochs 30 --batch_size 64 \
    --ckpt_dir {PROJECT_DIR}/ckpts/fusion_plainbce --resume


In [ ]:
# Compare final low-VAF recall across all 4 runs
import json, glob

print(f"{'run':30s} {'low_vaf_recall':>15s} {'overall_f1':>12s}")
for run_dir in ['fusion_vafaware', 'cnn_only', 'transformer_only', 'fusion_plainbce']:
    log_path = f'{PROJECT_DIR}/ckpts/{run_dir}/log.jsonl'
    if not os.path.exists(log_path):
        continue
    lines = [json.loads(l) for l in open(log_path)]
    best = max(lines, key=lambda r: (r['val']['low_vaf_recall'] == r['val']['low_vaf_recall'], r['val']['low_vaf_recall']))
    print(f"{run_dir:30s} {best['val']['low_vaf_recall']:15.3f} {best['val']['overall']['f1']:12.3f}")


## 10. Grad-CAM check on the best fusion checkpoint

Sanity-check that explainability is actually wired to real predictions, not
just the synthetic smoke test.


In [ ]:
import torch, matplotlib.pyplot as plt
from models import ExplainSomaticModel
from real_data import CachedSomaticDataset
from gradcam import grad_cam

model = ExplainSomaticModel(transformer_layers=6)
state = torch.load(f'{PROJECT_DIR}/ckpts/fusion_vafaware/best.pt', map_location='cpu')
model.load_state_dict(state['model_state'])
model.eval()

val_ds = CachedSomaticDataset(f'{CACHE_DIR}/val.h5')
pileup, ctx, label, vaf = val_ds[0]
heatmap = grad_cam(model, pileup.unsqueeze(0), ctx.unsqueeze(0))

plt.figure(figsize=(6, 10))
plt.imshow(heatmap.numpy(), aspect='auto', cmap='inferno')
plt.title(f'Grad-CAM | true label={label.item()}  true VAF={vaf.item():.3f}')
plt.xlabel('position in pileup window')
plt.ylabel('read index')
plt.colorbar(label='attribution')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/gradcam_example.png', dpi=150)
plt.show()


## 11. Scaling up beyond chr21

Once the chr21 run looks correct end to end:

1. Re-run steps 6–7 with `--region chr1`, `chr2`, ... (or a comma-separated
   region list if you extend `candidate_generation.py`'s argparse) to build
   caches for more chromosomes, then concatenate the HDF5 files or point the
   `CachedSomaticDataset` at a directory of shards (small extension to
   `real_data.py` if you want this — currently it takes one `.h5` per split).
2. Increase `--batch_size` if GPU memory allows (T4: ~64–96 is usually safe
   at this tensor size; A100 if your compute units stretch to one: much higher).
3. Everything else — model code, loss, checkpointing, Grad-CAM — is already
   scale-agnostic; nothing above is chr21-specific except the `--region` args.
